# AusLAMP Victorian MT Data: Re-Export Problem Sites

**Goal**: Attempt to re-export problematic sites to daily ASCII files with detailed error handling.

**Problem sites identified from batch run**:
- VIC046: Only 20,400 samples (0.0 days)
- VIC049: Length mismatch error
- VIC054: Only 1,800 samples
- VIC059: Only 13,200 samples
- VIC060: Only 19,200 samples
- VIC061: Only 12,600 samples
- VIC063: Length mismatch error
- VIC068: Length mismatch error
- VIC074: Only 5,400 samples
- VIC076: Length mismatch error
- VIC078b: Length mismatch error
- VIC100: Only 600 samples

**Output**: Daily ASCII files in `E:\MT_Timeseries_DATA\MT_AusLAMP_GA\EDL_2026\{site}\`

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
from obspy import read
from IPython.display import Markdown, display

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Pandas display options
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.2f}'.format)

def caption(text):
    """Display a figure caption as wrapped, italic markdown."""
    display(Markdown(f"*{text}*"))

## Custom Reader with Detailed Error Handling

In [2]:
def read_edl_site_robust(station_dir, verbose=True):
    """
    Read EDL MiniSEED data with robust error handling.
    
    Returns:
    - df: DataFrame with time series data (or None if failed)
    - issues: List of issues encountered
    """
    station_id = station_dir.name
    issues = []
    
    if verbose:
        print(f"\n{'='*80}")
        print(f"Reading: {station_id}")
        print(f"{'='*80}")
    
    # Scan for day folders
    day_folders = sorted([d for d in station_dir.glob('*') if d.is_dir()])
    
    if not day_folders:
        issues.append('NO_DAY_FOLDERS')
        if verbose:
            print(f"✗ ERROR: No day folders found")
        return None, issues
    
    if verbose:
        print(f"Found {len(day_folders)} day folders")
    
    # Collect all data by channel
    channels = ['BX', 'BY', 'BZ', 'EX', 'EY', 'TP']
    channel_data = {ch: [] for ch in channels}
    channel_times = {ch: [] for ch in channels}
    file_counts = {ch: 0 for ch in channels}
    file_errors = {ch: 0 for ch in channels}
    
    # Bartington Mag-03 calibration factors
    cal_bx = 0.007      # nT/µV for Bx
    cal_by = 0.007      # nT/µV for By
    cal_bz = 0.0175     # nT/µV for Bz
    
    for day_folder in day_folders:
        for ch in channels:
            # Find all files for this channel in this day
            files = sorted(list(day_folder.glob(f'*.{ch}')))
            
            for file_path in files:
                try:
                    # Read MiniSEED file
                    st = read(str(file_path), format='MSEED')
                    
                    if len(st) == 0:
                        issues.append(f'{ch}_EMPTY_FILE:{file_path.name}')
                        file_errors[ch] += 1
                        continue
                    
                    tr = st[0]
                    
                    # Extract data and times
                    data = tr.data.astype(np.float64)
                    times = tr.times('timestamp')
                    
                    # Apply calibration to magnetic channels
                    if ch == 'BX':
                        data = data * cal_bx
                    elif ch == 'BY':
                        data = data * cal_by
                    elif ch == 'BZ':
                        data = data * cal_bz
                    # EX, EY, TP stay as µV or ADC counts
                    
                    channel_data[ch].extend(data)
                    channel_times[ch].extend(times)
                    file_counts[ch] += 1
                    
                except Exception as e:
                    issues.append(f'{ch}_READ_ERROR:{file_path.name}:{str(e)}')
                    file_errors[ch] += 1
                    if verbose:
                        print(f"  ✗ Error reading {file_path.name}: {e}")
    
    # Report file counts
    if verbose:
        print(f"\nFiles read per channel:")
        for ch in channels:
            status = '✓' if file_errors[ch] == 0 else '⚠'
            print(f"  {status} {ch}: {file_counts[ch]} files, {len(channel_data[ch]):,} samples, {file_errors[ch]} errors")
    
    # Check for missing channels
    missing_channels = [ch for ch in channels if len(channel_data[ch]) == 0]
    if missing_channels:
        missing_str = ','.join(missing_channels)
        issues.append(f'MISSING_CHANNELS:{missing_str}')
        if verbose:
            print(f"\n✗ WARNING: Missing channels: {', '.join(missing_channels)}")
    
    # Check for length mismatches
    lengths = {ch: len(channel_data[ch]) for ch in channels if len(channel_data[ch]) > 0}
    if len(set(lengths.values())) > 1:
        issues.append(f'LENGTH_MISMATCH:{lengths}')
        if verbose:
            print(f"\n✗ WARNING: Sample count mismatch across channels:")
            for ch, length in lengths.items():
                print(f"  {ch}: {length:,} samples")
        
        # Truncate to minimum length
        min_length = min(lengths.values())
        if verbose:
            print(f"\n→ Truncating all channels to {min_length:,} samples")
        
        for ch in channels:
            if len(channel_data[ch]) > min_length:
                channel_data[ch] = channel_data[ch][:min_length]
                channel_times[ch] = channel_times[ch][:min_length]
    
    # Use the first available channel's timestamps
    primary_channel = None
    for ch in channels:
        if len(channel_times[ch]) > 0:
            primary_channel = ch
            break
    
    if primary_channel is None:
        issues.append('NO_DATA')
        if verbose:
            print(f"\n✗ ERROR: No data could be read from any channel")
        return None, issues
    
    # Convert to DataFrame
    timestamps = pd.to_datetime(channel_times[primary_channel], unit='s', utc=True)
    
    df_dict = {}
    for ch in channels:
        if len(channel_data[ch]) > 0:
            df_dict[ch] = channel_data[ch]
        else:
            # Fill missing channel with NaN
            df_dict[ch] = [np.nan] * len(timestamps)
    
    df = pd.DataFrame(df_dict, index=timestamps)
    
    if verbose:
        print(f"\n✓ Created DataFrame: {len(df):,} samples")
        print(f"  Time range: {df.index[0]} to {df.index[-1]}")
        print(f"  Duration: {(df.index[-1] - df.index[0]).total_seconds() / 86400:.2f} days")
    
    return df, issues

## Re-Export Problem Sites

In [3]:
# Base directories
base_dir = Path(r'E:\MT_Timeseries_DATA\MT_AusLAMP_GA\EDL_raw')
output_base = Path(r'E:\MT_Timeseries_DATA\MT_AusLAMP_GA\EDL_2026')
output_base.mkdir(exist_ok=True)

# Problem sites
problem_sites = [
    'VIC046', 'VIC049', 'VIC054', 'VIC059', 'VIC060', 'VIC061',
    'VIC063', 'VIC068', 'VIC074', 'VIC076', 'VIC078b', 'VIC100'
]

# Track results
export_results = []

print(f"Re-exporting {len(problem_sites)} problem sites...\n")
print(f"Output directory: {output_base}\n")

for i, site_id in enumerate(problem_sites):
    station_dir = base_dir / site_id
    
    if not station_dir.exists():
        print(f"[{i+1}/{len(problem_sites)}] {site_id}: DIRECTORY NOT FOUND\n")
        export_results.append({
            'station_id': site_id,
            'status': 'DIR_NOT_FOUND',
            'n_samples': 0,
            'n_days_exported': 0,
            'issues': ['DIRECTORY_NOT_FOUND']
        })
        continue
    
    start_time = time.time()
    
    try:
        # Read with robust error handling
        df, issues = read_edl_site_robust(station_dir, verbose=True)
        
        if df is None:
            print(f"\n[{i+1}/{len(problem_sites)}] {site_id}: FAILED TO LOAD\n")
            export_results.append({
                'station_id': site_id,
                'status': 'LOAD_FAILED',
                'n_samples': 0,
                'n_days_exported': 0,
                'issues': issues
            })
            continue
        
        elapsed = time.time() - start_time
        
        # Export to daily ASCII files
        station_output_dir = output_base / site_id
        station_output_dir.mkdir(exist_ok=True)
        
        print(f"\nExporting to daily files...")
        
        # Group by day
        df_export = df.copy()
        df_export['date'] = df_export.index.date
        days = df_export.groupby('date')
        
        n_days_exported = 0
        export_issues = []
        
        for date, day_df in days:
            try:
                # Get start time of this day's data
                day_start = day_df.index[0]
                
                # Format: YYYYMMDDHHMMSS.dat
                filename = day_start.strftime('%Y%m%d%H%M%S') + '.dat'
                filepath = station_output_dir / filename
                
                # Select time series channels only (BX, BY, BZ, EX, EY)
                # Drop TP and date columns
                export_df = day_df[['BX', 'BY', 'BZ', 'EX', 'EY']].copy()
                
                # Add timestamp as first column
                export_df.insert(0, 'DateTime', export_df.index.strftime('%Y-%m-%d %H:%M:%S.%f'))
                
                # Write to file
                export_df.to_csv(filepath, sep=' ', index=False, float_format='%.6f')
                n_days_exported += 1
                
            except Exception as e:
                export_issues.append(f'EXPORT_ERROR:{date}:{str(e)}')
                print(f"  ✗ Error exporting {date}: {e}")
        
        print(f"\n✓ Exported {n_days_exported} day files")
        print(f"  Processing time: {elapsed:.1f}s")
        
        all_issues = issues + export_issues
        status = 'SUCCESS' if len(all_issues) == 0 else 'SUCCESS_WITH_WARNINGS'
        
        export_results.append({
            'station_id': site_id,
            'status': status,
            'n_samples': len(df),
            'duration_days': (df.index[-1] - df.index[0]).total_seconds() / 86400,
            'n_days_exported': n_days_exported,
            'processing_time': elapsed,
            'issues': all_issues
        })
        
    except Exception as e:
        print(f"\n✗ UNEXPECTED ERROR: {e}\n")
        export_results.append({
            'station_id': site_id,
            'status': 'EXCEPTION',
            'n_samples': 0,
            'n_days_exported': 0,
            'issues': [f'EXCEPTION:{str(e)}']
        })

print(f"\n{'='*80}")
print(f"Re-export completed!")
print(f"{'='*80}")

Re-exporting 12 problem sites...

Output directory: E:\MT_Timeseries_DATA\MT_AusLAMP_GA\EDL_2026


Reading: VIC046
Found 22 day folders

Files read per channel:
  ✓ BX: 480 files, 17,280,000 samples, 0 errors
  ✓ BY: 480 files, 17,280,000 samples, 0 errors
  ✓ BZ: 480 files, 17,280,000 samples, 0 errors
  ✓ EX: 480 files, 17,280,000 samples, 0 errors
  ✓ EY: 480 files, 17,280,000 samples, 0 errors
  ✓ TP: 480 files, 17,280,000 samples, 0 errors

✓ Created DataFrame: 17,280,000 samples
  Time range: 2014-05-20 01:00:00+00:00 to 2014-06-09 00:59:59.900000095+00:00
  Duration: 20.00 days

Exporting to daily files...

✓ Exported 21 day files
  Processing time: 96.1s

Reading: VIC049
Found 43 day folders

Files read per channel:
  ✓ BX: 998 files, 35,928,000 samples, 0 errors
  ✓ BY: 998 files, 35,928,000 samples, 0 errors
  ✓ BZ: 998 files, 35,928,000 samples, 0 errors
  ✓ EX: 998 files, 35,928,000 samples, 0 errors
  ✓ EY: 998 files, 35,928,000 samples, 0 errors
  ✓ TP: 998 files, 35,928,

## Summary Report

In [4]:
# Convert to DataFrame
results_df = pd.DataFrame(export_results)

print("\n" + "="*80)
print("EXPORT SUMMARY")
print("="*80 + "\n")

# Count by status
status_counts = results_df['status'].value_counts()
print("Status breakdown:")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

print(f"\nTotal sites attempted: {len(results_df)}")
print(f"Successfully exported: {len(results_df[results_df['status'].str.contains('SUCCESS')])}")
print(f"Failed: {len(results_df[~results_df['status'].str.contains('SUCCESS')])}")

# Show full results table
print("\n" + "="*80)
print("DETAILED RESULTS")
print("="*80 + "\n")

display(results_df)


EXPORT SUMMARY

Status breakdown:
  SUCCESS: 11
  SUCCESS_WITH_WARNINGS: 1

Total sites attempted: 12
Successfully exported: 12
Failed: 0

DETAILED RESULTS



,station_id,status,n_samples,duration_days,n_days_exported,processing_time,issues
0,VIC046,SUCCESS,17280000,20.00,21,96.12,[]
1,VIC049,SUCCESS,35928000,41.58,42,205.29,[]
2,VIC054,SUCCESS,28405800,32.92,34,162.34,[]
3,VIC059,SUCCESS,14289600,16.57,17,81.00,[]
4,VIC060,SUCCESS,12223200,14.17,15,90.19,[]
5,VIC061,SUCCESS,13008600,15.08,16,92.86,[]
6,VIC063,SUCCESS,12199220,16.08,16,85.97,[]
7,VIC068,SUCCESS,29441400,34.08,35,164.74,[]
8,VIC074,SUCCESS,7997400,9.29,10,46.52,[]
9,VIC076,SUCCESS_WITH_WARNINGS,12180600,14.12,15,67.02,[BY_READ_ERROR:VIC076_140628150000.BY:julday o...


## Sites That Still Need Work

In [5]:
# Filter failed sites
failed_sites = results_df[~results_df['status'].str.contains('SUCCESS')]

if len(failed_sites) > 0:
    print(f"\n{len(failed_sites)} sites still have issues:\n")
    
    for idx, row in failed_sites.iterrows():
        print(f"\n{row['station_id']}:")
        print(f"  Status: {row['status']}")
        print(f"  Issues:")
        for issue in row['issues']:
            print(f"    - {issue}")
else:
    print("\n✓ All sites successfully exported!")


✓ All sites successfully exported!


## Sites With Warnings

In [6]:
# Sites that exported but had issues
warning_sites = results_df[results_df['status'] == 'SUCCESS_WITH_WARNINGS']

if len(warning_sites) > 0:
    print(f"\n{len(warning_sites)} sites exported with warnings:\n")
    
    for idx, row in warning_sites.iterrows():
        print(f"\n{row['station_id']}:")
        print(f"  Samples: {row['n_samples']:,}")
        print(f"  Days exported: {row['n_days_exported']}")
        print(f"  Duration: {row['duration_days']:.2f} days")
        print(f"  Issues:")
        for issue in row['issues']:
            print(f"    - {issue}")
else:
    print("\n✓ No warnings!")


1 sites exported with warnings:


VIC076:
  Samples: 12,180,600
  Days exported: 15
  Duration: 14.12 days
  Issues:
    - BY_READ_ERROR:VIC076_140628150000.BY:julday out of bounds (wrong endian?): 0
    - LENGTH_MISMATCH:{'BX': 12216600, 'BY': 12180600, 'BZ': 12216600, 'EX': 12216600, 'EY': 12216600, 'TP': 12216600}


## Save Results to CSV

In [7]:
# Convert issues list to string for CSV
results_df['issues_str'] = results_df['issues'].apply(lambda x: '; '.join(x) if isinstance(x, list) else str(x))

# Save to outputs directory
output_dir = Path.cwd().parent / 'outputs'
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / 'problem_sites_reexport_results.csv'
results_df.to_csv(csv_path, index=False)

print(f"✓ Results saved to: {csv_path}")
print(f"  {len(results_df)} sites, {len(results_df.columns)} columns")

✓ Results saved to: d:\BK_AusLAMP_GA\outputs\problem_sites_reexport_results.csv
  12 sites, 8 columns
